# Tiled Attention — Solution

Reference implementation for `02_tiled_attention_exercise.ipynb`.

## Setup

In [1]:
import torch

## Solution 1 — `flash_attention`

In [2]:
def flash_attention(Q, K, V, block_size=64):
    seq_len,d = Q.shape  
    attention_out = torch.zeros(Q.shape)

    for i in range(0,Q.shape[0],block_size):
        Q_chunk = Q[i:i+block_size,:]  ##Block_size,d

        O_i = torch.zeros(Q_chunk.shape)
        m_i = torch.full((Q_chunk.shape[0],), float("-inf"))
        l_i = torch.zeros(Q_chunk.shape[0])

        for j in range(0,K.shape[0],block_size):
            K_chunk = K[j:j+block_size,:]
            V_chunk = V[j:j+block_size,:]

            scores = Q_chunk@K_chunk.T / d**0.5  ##block_size,block_size
            chunk_max,_ = torch.max(scores,dim=-1) ## (block_size,)
            m_new = torch.maximum(m_i,chunk_max)
            correction_factor = torch.exp(m_i-m_new)
            
            l_i = l_i * correction_factor + torch.exp(scores-m_new[:,None]).sum(dim=-1)
            O_i = O_i * correction_factor[:,None] + torch.exp(scores-m_new[:,None]) @ V_chunk

            m_i = m_new

        O_i = O_i / l_i[:,None]
        attention_out[i:i+block_size,:] = O_i

    return attention_out

In [3]:
Q, K, V  = torch.randn((256,64)),torch.randn((256,64)),torch.randn((256,64))
flash_attention(Q, K, V, block_size=64)

tensor([[-0.0673, -0.0882,  0.0543,  ...,  0.0252,  0.0466, -0.0169],
        [-0.0140, -0.0719,  0.0597,  ..., -0.0905, -0.0276,  0.1022],
        [ 0.0497, -0.0236,  0.0623,  ...,  0.1269,  0.0541,  0.1594],
        ...,
        [ 0.1116, -0.0614, -0.0026,  ..., -0.0864,  0.0335, -0.0391],
        [ 0.1891,  0.0024,  0.0928,  ...,  0.0299, -0.1001,  0.1476],
        [-0.0384, -0.1218,  0.0089,  ..., -0.0916, -0.0997, -0.2091]])

**Why outer-Q, inner-KV (FA2) instead of FA1's outer-KV?** The partial output `O_i` accumulates across all K, V blocks for one Q block. Putting Q in the outer loop means `O_i` lives in SRAM the entire time — only one HBM write at the end. With Q inner (FA1), each visit to a Q block rewrites its partial state to HBM, then reloads it next time. The desk/notebook analogy from the blog._

**Why this is exact.** _Online softmax's correction factor is mathematically identical to batch softmax — no approximation. Combined with the standard matmul tiling for QK^T and PV, the entire algorithm preserves the exact attention math._

## Run the tests

In [4]:
from tests import run_flash_attention_tests
run_flash_attention_tests(flash_attention)

Running flash_attention...
  ✓ Matches reference (non-causal)
  ✓ Output independent of block_size

  All 2 checks passed ✓



True